# ¿Cuánto Netflix vemos realmente?

**Autor: Roberto Cantero · Nebula · EDA · 20 de noviembre de 2026**

## Mi pregunta

Tengo miles de títulos para elegir, pero quiero saber cuánto de las horas vistas se concentra en unos pocos. Trabajo con el informe mundial de enero-junio de 2026. No identifico personas ni genero predicciones.

## Mi ruta

Primero preparo los registros y compruebo sus unidades; en el segundo cuaderno analizo el reparto. Puedo ejecutar ambos sin internet y sin instalar dependencias, con Python 3.10 o posterior. Mantengo el Excel original y preparar_datos.py junto a estos cuadernos.

In [1]:
# leo mi copia original y separo las filas de datos de las notas
from pathlib import Path
from preparar_datos import preparar
import csv
import hashlib
import json
from collections import Counter
raiz = Path.cwd()
registros = preparar()
print('Filas numéricas:', len(registros))
print('SHA-256:', hashlib.sha256((raiz/'netflix_2026_s1.xlsx').read_bytes()).hexdigest())

Filas numéricas: 17372
SHA-256: bcfda87e87d09d923472a1e1b30d5c7c799457aacba41310a9d223e834fe2f5f


## Mis unidades y decisiones

Conservo cada fila tal como la publica Netflix: una película, una temporada u otro programa. No agrupo temporadas. El nombre no es un identificador único: puedo encontrar películas distintas con el mismo título. No las fusiono.

Separo Other Movies y Other Shows de las entradas identificadas. Mantengo sus horas en el denominador, porque forman parte del tiempo total. No los trato como dos títulos. El informe indica que estas bolsas recogen títulos con menos de 50.000 visualizaciones; los videopódcast se incluyen en Other Shows. Las visualizaciones no publicadas de las bolsas quedan como valores ausentes, nunca como cero.

In [2]:
# compruebo mis unidades y conservo los agregados
otros = [r for r in registros if r['titulo'] in {'Other Movies', 'Other Shows'}]
identificados = [r for r in registros if r not in otros]
assert len(otros) == 2
assert all(r['horas'] >= 0 for r in registros)
assert all(isinstance(r['visualizaciones'], int) for r in identificados)
assert sum(r['horas'] for r in registros) == 97664500000
assert len(identificados) == 17370
print('Entradas identificadas:', len(identificados))
print('Horas en otras agrupaciones:', sum(r['horas'] for r in otros))
print('Entradas por formato:', dict(Counter(r['formato'] for r in identificados)))
repetidos = {k:v for k,v in Counter((r['formato'],r['titulo']) for r in identificados).items() if v>1}
print('Nombres repetidos conservados:', repetidos)
print('Notas originales:', json.loads((raiz/'notas_originales.json').read_text(encoding='utf-8')))

Entradas identificadas: 17370
Horas en otras agrupaciones: 859900000
Entradas por formato: {'series': 8193, 'peliculas': 9177}
Nombres repetidos conservados: {('peliculas', 'War Machine'): 2, ('peliculas', 'Monster'): 2, ('peliculas', 'Maria'): 2}
Notas originales: {'series': [{'A': '*“Other Movies” and “Other Shows” rows in this report capture total viewing from titles with <50k views. Additionally, all video podcasts are calculated under “Other Shows”. These rows allow the report to reflect 100% of viewing. “Other” in 2025 H1 totalled 662,600,000 view hours.'}], 'peliculas': [{'A': '*“Other Movies” and “Other Shows” rows in this report capture total viewing from titles with <50k views. Additionally, all video podcasts are calculated under “Other Shows”. These rows allow the report to reflect 100% of viewing. “Other” in 2025 H1 totalled 662,600,000 view hours.'}]}


## Lo que dejo preparado

He generado netflix_2026_s1.csv sin cambiar las cifras originales. No confundo el listado mundial de consumo con el catálogo disponible para un usuario. Las horas están redondeadas: las cifras derivadas también son aproximadas. No deduzco ausencias o cero consumo a partir de las obras que no aparecen identificadas. Continúo en 02_eda.ipynb.

## Mis fuentes

- [Presentación oficial y descarga original](https://about.netflix.com/en/news/what-we-watched-the-first-half-of-2026).
- Excel netflix_2026_s1.xlsx, hojas Shows y Movies, incluida su nota al pie.
- [Contexto periodístico que motivó la pregunta, 19 de noviembre](https://www.digitaltrends.com/movies/netflix-has-325-million-subscribers-but-its-ceo-thinks-the-real-battle-is-just-beginning/). El análisis usa las cifras del archivo oficial, no las de esta noticia.